[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_Mach_Learn/Uncertainty_in_ML.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Uncertainty in ML

A model that says '87%' should be right 87% of the time — most deep models aren't. Two sessions on measuring calibration and fixing it with the workhorse tools: temperature scaling and deep ensembles, with a hard look at what happens *off*-distribution.

## 1. Pre-requisites

- [Training Dynamics](./Training_Dynamics.ipynb) (we reuse its spiral testbed).
- [Kernel Methods](./Kernel_Methods.ipynb) S2 — GPs as the calibration gold standard.
- [Estimation Theory](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb) S3 — the Bayesian frame.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as Fn
import matplotlib.pyplot as plt
torch.manual_seed(0); rng = np.random.default_rng(0)

# noisy spirals again — genuine class overlap means genuine aleatoric uncertainty
def spirals(n=3000, noise=0.9, seed=0):
    r = np.random.default_rng(seed)
    t = np.linspace(0.5, 3*np.pi, n//2)
    X, y = [], []
    for cls, ph in [(0, 0.0), (1, np.pi)]:
        X.append(np.stack([t*np.cos(t+ph), t*np.sin(t+ph)], 1) + noise*r.standard_normal((n//2, 2)))
        y.append(np.full(n//2, cls))
    X = np.concatenate(X).astype(np.float32); y = np.concatenate(y).astype(np.int64)
    X = (X - X.mean(0)) / X.std(0)
    idx = r.permutation(n)
    return torch.from_numpy(X[idx]), torch.from_numpy(y[idx])

Xa, ya = spirals()
Xtr, ytr, Xte, yte = Xa[:2000], ya[:2000], Xa[2000:], ya[2000:]

def make_net(seed):
    torch.manual_seed(seed)
    return nn.Sequential(nn.Linear(2, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 2))

def fit(net, epochs=600):     # deliberately overtrained — watch the confidence outrun the accuracy
    opt = torch.optim.Adam(net.parameters(), lr=2e-3)
    for ep in range(epochs):
        for i in range(0, 2000, 200):
            opt.zero_grad()
            Fn.cross_entropy(net(Xtr[i:i+200]), ytr[i:i+200]).backward()
            opt.step()
    return net

---
### 🕐 Session 1 of 2 — *Calibration & Temperature Scaling* (~40 min)
**Goal:** measure whether confidences mean anything; fix miscalibration with one parameter.
**Builds on:** [Training Dynamics](./Training_Dynamics.ipynb). &nbsp; **Feeds into:** Session 2 (ensembles & OOD).

---

## 2. Does 87% Mean 87%?

💡 **Intuition.** Accuracy asks 'how often right?'; **calibration** asks 'when you say 87%, are you right 87% of the time?' The reliability diagram answers it: bin predictions by confidence, plot accuracy per bin against the diagonal. Deep nets trained to convergence sit *below* the diagonal — overconfident — because cross-entropy keeps rewarding sharper probabilities long after accuracy saturates. The embarrassingly effective fix: **temperature scaling** — divide the logits by one scalar $T$ fitted on validation data. It can't change any decision (argmax is $T$-invariant); it only makes the *confidence honest*.

In [ ]:

# YOUR CODE HERE


**What just happened.** Two numbers that answer two different questions: **accuracy 89.8%** — how often the model is right — and **ECE 0.041** — whether the confidence it prints means anything. On average the stated confidence is off by about **4 percentage points**, and for a deep network trained to convergence the error is in the expected direction: overconfident.

**Read ECE as what it literally computes.** Bin the test predictions by stated confidence; in each bin compare mean confidence against actual accuracy; average the gaps weighted by bin population. So 0.041 means "when this model says 90%, it is right about 86% of the time" — small enough to sound harmless, large enough to matter if you are thresholding on confidence to decide whether a human should review the case.

**The overconfidence is manufactured by the training loop, not by bad luck.** Cross-entropy keeps rewarding sharper probabilities after accuracy has saturated: moving a correct prediction from 0.90 to 0.99 lowers the loss even though it changes no decision. `fit` runs **600 epochs deliberately** so the confidence has time to outrun the accuracy. This is the well-documented modern failure mode — accuracy improved across a decade of architecture research while calibration got *worse*.

**But be careful about how much of the 0.041 is fixable.** This spiral is generated with `noise=0.9`, so the classes genuinely overlap and a large share of the model's uncertainty is **aleatoric** — irreducible label noise that no amount of calibration removes. A perfectly calibrated model on this data would still be wrong about 10% of the time and would *correctly* say so. Temperature scaling can only address the part that comes from overconfidence, which is why the improvement in the next cell is modest rather than dramatic.

**Note two limitations of ECE before quoting it anywhere.** It is **binning-dependent** — 10 bins and 15 bins give different numbers for identical predictions — and it is an **average**, so a model can post a small ECE while being badly miscalibrated in exactly the high-confidence bin that drives automated decisions. The `reliability` helper here also drops bins with fewer than 5 points, which quietly discards sparse regions. **Look at the diagram, not only the scalar.**

**Finally, keep the two numbers separate in your head, because they trade off independently.** A model can be accurate and dishonest (right often, always claiming 99%) or honest and weak (right 60% of the time and saying 60%). For a system that hands borderline cases to a human, the second is deployable and the first is dangerous. **Accuracy tells you how good the model is; calibration tells you whether you can act on what it says.**

In [ ]:
# fit T on a validation split by minimizing NLL — one parameter, no retraining

# YOUR CODE HERE


**What just happened.** Two reliability curves against the dashed diagonal. The raw curve sits **below** it — the model claims more confidence than it earns — and the temperature-scaled curve is dragged toward it, with a lower ECE in the legend. One scalar, fitted by LBFGS on 500 held-out points, no retraining.

**The property that makes temperature scaling remarkable is that it cannot cost you anything.** Softmax is monotone and $T > 0$, so dividing every logit by $T$ **never changes the argmax**. Accuracy is preserved *exactly* — not approximately, not on average, but example by example. All that changes is how sharp the probabilities are. A post-hoc fix with one parameter and a guarantee attached is rare; take it whenever you have a validation split.

**Note where $T$ was fitted and where it was evaluated, because this is the one way to get it wrong.** $T$ comes from `logits[:500]`; both curves are computed on `logits[500:]`. Fit and report on the same data and any improvement is circular. Students copying this pattern get it wrong constantly, and the fix is the two slices visible in the code.

**Read $T$ itself as a diagnostic.** $T > 1$ softens the logits and means the model was **overconfident**; $T < 1$ sharpens them and means it was underconfident. Trained-to-convergence networks essentially always land above 1 — cross-entropy keeps rewarding sharper probabilities long after accuracy has stopped improving, so 600 epochs buy confidence rather than correctness.

**Be honest that the improvement here is modest, and say why.** The starting ECE is 0.041, which is mild to begin with. The spiral is generated with `noise=0.9`, so the classes genuinely overlap and much of the model's uncertainty is **aleatoric** — irreducible label noise that is *correct* to report and that no calibration method should remove. Temperature scaling only addresses the overconfident component. On a ResNet trained on CIFAR you would typically see ECE fall from 0.10–0.15 to under 0.02 and the raw curve bow visibly away from the diagonal; this demo shows the mechanism on a problem that was not badly broken.

**And be clear about the fundamental limitation, which Session 2 exists to probe.** A single global $T$ applies the same correction everywhere in input space. It cannot say "I am well calibrated near the training data and hopeless 5 units away" — that would require the correction to depend on $x$. **Temperature scaling fixes the *average* honesty of a model on the distribution it was calibrated on, and does nothing off it.** Every confidence number in this notebook, raw or scaled, is conditional on the test point coming from the same world as the training data.

---
### 🕐 Session 2 of 2 — *Ensembles & the Out-of-Distribution Problem* (~40 min)
**Goal:** average independently-trained nets for better uncertainty; test where all bets are off.
**Builds on:** Session 1.

---

## 3. Deep Ensembles

💡 **Intuition.** Train the same architecture from $k$ different random seeds and *average the probabilities*. Where the data constrains the function, the members agree; where it doesn't, they disagree — and that **disagreement is an uncertainty signal** that single-model confidence simply doesn't carry. It's a crude Bayesian posterior ([Estimation Theory S3](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb)), cousin to the [particle filter](../Intro_Time_Series/Beyond_Kalman.ipynb): represent belief with samples. Cost: $k\times$ everything — and it remains the strongest practical baseline in the field.

In [ ]:

# YOUR CODE HERE


**What just happened.** Five independently trained networks, averaged — and **nothing improved**:

| | accuracy | ECE |
|---|---|---|
| single model | 89.8% | 0.041 |
| 5-ensemble | **89.4%** | **0.041** |

Five times the training cost, five times the inference cost, and the accuracy is slightly *lower* while the calibration error is identical. **This is a null result, and it is worth reporting as one.**

**The cause is diagnosable, and it is in the code.** `fit_boot` draws a **bootstrap resample** — `torch.randint(0, 2000, (2000,))` — so each member sees only about $1 - e^{-1} \approx 63\%$ of the unique training points. Bootstrapping buys diversity between members and pays for it in data per member. On a task already sitting near its aleatoric floor, the data lost costs about as much as the averaging gains, and the two effects cancel. Drop the bootstrap and train five members on the full set with different seeds only, and the accuracy typically edges up while diversity drops.

**The 0.4-point accuracy difference is also within noise.** On 1000 test points the binomial standard error is about 1%, so 89.8% versus 89.4% is well under half a standard error — four examples. **Do not read a ranking into it in either direction.** The honest statement is that on this task, at this scale, the ensemble bought nothing measurable.

**So say what ensembles are actually for, because it is not this.** In-distribution calibration is temperature scaling's job: one parameter, no retraining, provably no accuracy cost. Paying 5× to match it is a bad trade. **Ensembles earn their keep on the problem temperature scaling structurally cannot touch** — behaviour *off* the training distribution, where a single model has no mechanism for doubt at all. The next cell is where that shows up.

**And keep the mechanism in view, since it is what makes the off-distribution case different.** A single softmax outputs one number and has no way to express "the data does not determine this". Five models that agree where the data constrains them and diverge where it does not carry an extra signal — **disagreement** — that no single model can produce, at any confidence value. This cell shows that the signal is worthless where the data is dense; the next shows where it becomes the only thing you have.

**One framing worth keeping.** This is a crude Bayesian posterior represented by samples — the same idea as the [particle filter](../Intro_Time_Series/Beyond_Kalman.ipynb) in the time-series track, where belief is carried by a population rather than a formula. Five samples is a very coarse posterior, which is part of why the next result is only partly good news.

## 4. Off the Map

💡 **Intuition.** The dirty secret of every confidence score: it is only meaningful **on the distribution the model was trained on**. Show the network a point from a different world and softmax still prints a confident number — softmax *must* sum to one; it has no 'none of the above'. Ensemble disagreement (predictive entropy) at least *rises* off-distribution. Compare both on points far outside the spiral:

In [ ]:
# scan the plane: single-model confidence vs ensemble entropy
# find the off-map points the ensemble actually flags (and admit the ones it doesn't)

# YOUR CODE HERE


**What just happened.** The plane, painted twice. Single-model max-softmax is **confident almost everywhere**, including the far corners where no training point has ever been. Ensemble entropy rises in some off-spiral regions — visible improvement — and then the numbers arrive and complicate the story:

- off-map area (radius > 3) flagged by ensemble entropy: **9%**
- off-map area where the ensemble *and* the single net are both confident: **90%**

**Take the good news first, because it is real.** At $(+3.1, +1.6)$ the single network reports **100.0%** confidence while the ensemble entropy is **0.69**, which is exactly $\log 2$ — the theoretical maximum for two classes, a perfect coin flip, a complete "I do not know". A single softmax has no way to produce that statement at any confidence value. Where the members disagree, the ensemble says so unambiguously.

**Now the bad news, which is the honest headline: 9% coverage.** Nine tenths of the off-map region is territory where **all five members agree confidently about a point none of them has any basis to judge**. Ensembles *mitigate* out-of-distribution overconfidence; they do not solve it, and a 9% detection rate would not pass as a safety mechanism anywhere.

**The mechanism explains the failure and is worth stating.** The five members share an architecture, a training set, a loss, and an inductive bias. Different seeds buy independence of *initialisation* — not independence of *assumption*. Far from the data all five extrapolate the same way, because ReLU networks extend their outermost linear pieces to infinity and those pieces are largely determined by the data, not the seed. **Disagreement requires the members to be wrong differently, and nothing forced them to be.**

**Underneath both panels is a structural fact about the output layer.** Softmax **must** sum to one. There is no "none of the above" class, so any input — a spiral point, a point 5 units away, a photograph, random noise — has its probability mass distributed among the two classes the model knows. **Confidence is a statement about which class, never about whether the question makes sense.** That is not a training failure to be fixed with more epochs; it is what the architecture computes.

**Which sets the practical rule this workshop exists to deliver.** Every confidence number — raw, temperature-scaled, or ensembled — is **conditional on the input being on-distribution**, and nothing in the model verifies that condition. Detecting "off the map" is its own problem with its own literature (density estimation, Mahalanobis distance in feature space, deep evidential methods, explicit OOD training), and ensemble disagreement is a first tool rather than an answer.

**One caveat on the numbers themselves.** The 9% and 90% figures depend on the arbitrary thresholds `ent > 0.35` and `ent < 0.05 & p > 0.99`, and on defining "off-map" as radius > 3.0. Move those and the percentages move. The qualitative conclusion — most of the off-map plane is confidently classified by both methods — is robust to any reasonable choice, and it is worth checking that yourself rather than taking the two printed numbers at face value.

**For contrast, recall the [GP](./Kernel_Methods.ipynb) from the kernel workshop.** Its posterior variance returns to the *prior* far from data, automatically, because the formula subtracts only what the data explains. That is uncertainty by construction rather than by disagreement — and even it audited at 92% coverage rather than 95%. Nobody in this area is doing better than usefully imperfect; knowing which imperfection you have is the skill.

## 5. Conclusion

Calibrate before you trust (temperature is nearly free); ensemble when the stakes justify 5×; and treat *all* confidences as conditional on being on-distribution — detecting 'off the map' is its own problem, and disagreement is your first tool — but as the area numbers show, it flags only *part* of the off-map world: ensembles mitigate OOD overconfidence, they do not solve it. The [GP](./Kernel_Methods.ipynb) remains the standard these methods chase.

---
## Where next

- [Kernel Methods](./Kernel_Methods.ipynb) S2 — calibrated uncertainty by construction.
- [Beyond Kalman](../Intro_Time_Series/Beyond_Kalman.ipynb) — belief-as-samples, the filtering version.
- [Estimation Theory](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb) — what 'well-calibrated' means formally.